# Практикалық жұмыс №5
## Тақырыбы: Тірек векторлық машиналар негізінде классификаторды жүзеге асыру (Support Vector Machines – SVM)
**Оқу бағдарламасы:** 6B06106 – Computer Science  
**Орындаған:** [Имя удалено по соображениям конфиденциальности]  
**Тексерген:** аға оқытушы [Имя удалено по соображениям конфиденциальности]  
**Тапсырма:** Студенттердің сабаққа қатысуы, оқу уақыты және ағымдағы тест көрсеткіштері негізінде емтиханнан өтуін болжау.

In [ ]:
# 1. Қажетті кітапханаларды жүктеу
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
# 2. Деректерді жүктеу және бастапқы құрылымын тексеру
df = pd.read_csv('data.csv')
print("Деректер өлшемі:", df.shape)
print("\n--- Алғашқы 5 жазба ---")
print(df.head())
print("\n--- Деректер құрылымы туралы ақпарат ---")
print(df.info())

In [ ]:
# 3. Статистикалық сипаттамалар және деректерді алдын ала талдау (EDA)
print("--- Сипаттамалық статистика ---")
print(df.describe())
print("\nБос мәндер саны (isnull().sum()):\n", df.isnull().sum())
print("\nҚайталанатын жазбалар саны:", df.duplicated().sum())
print("\nКластар үлестірімі (result):\n", df['result'].value_counts())

In [ ]:
# 4. Деректерді визуализациялау
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Шашырау диаграммасы: Сабаққа қатысу және тест нәтижесі
scatter = axes[0].scatter(df['attendance'], df['test_score'], c=df['result'], 
                          cmap='coolwarm', alpha=0.75, edgecolors='k', s=50)
axes[0].set_title('Сабаққа қатысу және тест балы бойынша үлестірім')
axes[0].set_xlabel('Сабаққа қатысу пайызы (attendance, %)')
axes[0].set_ylabel('Ағымдағы тест балы (test_score)')
axes[0].grid(True, linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=axes[0], ticks=[0, 1])
cbar.set_ticklabels(['Өтпеді (0)', 'Өтті (1)'])

# Ящиктік диаграмма: Оқу сағатының емтихан нәтижесіне байланысы
sns.boxplot(x='result', y='study_hours', data=df, ax=axes[1], hue='result', palette=['#6baed6', '#fb6a4a'], legend=False)
axes[1].set_title('Оқу сағатының емтихан нәтижесіне байланысы')
axes[1].set_xlabel('Мақсатты класс (0 - Өтпеді, 1 - Өтті)')
axes[1].set_ylabel('Оқуға жұмсалған апталық сағат (study_hours)')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Өтпеді (0)', 'Өтті (1)'])
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 5. Белгілер мен мақсатты айнымалыны анықтау, бөлу және масштабтау
features = ['attendance', 'study_hours', 'test_score']
X = df[features]
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Оқыту жиыны:", X_train_scaled.shape)
print("Тестілеу жиыны:", X_test_scaled.shape)

In [ ]:
# 6. Әртүрлі ядроларды (Linear, RBF, Poly) салыстыру
kernels = ['linear', 'rbf', 'poly']
comparison_data = []

for k in kernels:
    svc_model = SVC(kernel=k, C=1.0, random_state=42)
    svc_model.fit(X_train_scaled, y_train)
    p = svc_model.predict(X_test_scaled)
    comparison_data.append({
        'Ядро (Kernel)': k.capitalize(),
        'Accuracy': accuracy_score(y_test, p),
        'Precision': precision_score(y_test, p),
        'Recall': recall_score(y_test, p),
        'F1-score': f1_score(y_test, p),
        'Тірек векторлар саны': svc_model.support_vectors_.shape[0]
    })

comparison_df = pd.DataFrame(comparison_data)
print("--- Ядроларды салыстыру кестесі ---")
print(comparison_df.to_string(index=False))

In [ ]:
# 7. C регуляризация параметрінің әсерін зерттеу (RBF ядросы)
c_values = [0.1, 1.0, 10.0, 100.0]
c_metrics = []

for c in c_values:
    model_c = SVC(kernel='rbf', C=c, random_state=42)
    model_c.fit(X_train_scaled, y_train)
    p_c = model_c.predict(X_test_scaled)
    c_metrics.append({
        'C параметрі': c,
        'Accuracy': accuracy_score(y_test, p_c),
        'Тірек векторлар саны': model_c.support_vectors_.shape[0],
        '0-класс векторлары': model_c.n_support_[0],
        '1-класс векторлары': model_c.n_support_[1]
    })

print(pd.DataFrame(c_metrics).to_string(index=False))

In [ ]:
# 8. Негізгі модельді оқыту және сапасын кешенді бағалау (RBF, C=1.0)
best_svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
best_svm.fit(X_train_scaled, y_train)

y_pred = best_svm.predict(X_test_scaled)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1-score: ", f1_score(y_test, y_pred))
print("\n--- Тірек векторлар туралы ақпарат ---")
print(f"Жалпы саны: {best_svm.support_vectors_.shape[0]}")
print(f"0-класс бойынша (Өтпегендер): {best_svm.n_support_[0]}")
print(f"1-класс бойынша (Өткендер):   {best_svm.n_support_[1]}")
print("\n--- Classification Report ---\n", classification_report(y_test, y_pred, target_names=['Өтпеді (0)', 'Өтті (1)']))

In [ ]:
# 9. Confusion Matrix визуализациясы
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Өтпеді (0)', 'Өтті (1)'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - SVC (kernel="rbf", C=1.0)')
plt.show()

In [ ]:
# 10. Жаңа студенттер үшін емтихан нәтижесін болжау және ықтималдықты анықтау
new_students = pd.DataFrame([
    # 1-студент: Сабаққа жақсы қатысып, көп оқыған студент
    {'attendance': 88.0, 'study_hours': 6.5, 'test_score': 82.0},
    # 2-студент: Сабаққа сирек келіп, аз оқыған студент
    {'attendance': 42.0, 'study_hours': 1.5, 'test_score': 36.0}
], columns=features)

new_scaled = scaler.transform(new_students)
predictions = best_svm.predict(new_scaled)
probabilities = best_svm.predict_proba(new_scaled)

for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    status = "ЕМТИХАННАН ӨТЕДІ (1)" if pred == 1 else "ЕМТИХАННАН ӨТПЕЙДІ (0)"
    print(f"Студент #{i+1} бойынша болжам: {status}")
    print(f"  Ықтималдықтар: P(Өтпейді) = {prob[0]:.4f} ({prob[0]*100:.2f}%), P(Өтеді) = {prob[1]:.4f} ({prob[1]*100:.2f}%)\n")